# Core Drills — the ~12 problems actually worth drilling

**Easy 1–12** first: tiny one-concept finger exercises, the realistic screen openers.
Then the core problems, cut from notebooks 00–04 and ordered by likelihood for the **KoiReader systems-first CV** role.
Top 7 = drill until you can write them cold. Bottom 5 = plausible given KoiReader's OCR/doc-AI product.

Everything else in the original notebooks (deskew, Hough, template matching, CLAHE, barcode
morphology, E1–E8) is **skim-only** — better spent on GStreamer/threading prep.


In [ ]:
%matplotlib inline
import cv2, numpy as np, matplotlib.pyplot as plt, os
from collections import deque

def show(img, title=""):
    """Display a BGR or grayscale image inline."""
    if img is None:
        print("image is None"); return
    plt.figure(figsize=(5,5))
    if img.ndim == 2:
        plt.imshow(img, cmap="gray")
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title); plt.axis("off"); plt.show()

def make_sample_assets():
    """Create synthetic inputs so every cell runs without your own files.
    Swap input.jpg for a real photo any time to experiment."""
    doc = np.full((700,500,3), 30, np.uint8)
    quad = np.array([[120,90],[420,140],[400,560],[90,520]], np.int32)
    cv2.fillConvexPoly(doc, quad, (235,235,235))
    cv2.putText(doc, "LABEL 12345", (150,300), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (20,20,20), 2)
    cv2.imwrite("input.jpg", doc)
    cv2.imwrite("noisy.jpg", doc)
    cv2.imwrite("low.jpg", (doc*0.3 + 90).astype(np.uint8))
    os.makedirs("ind", exist_ok=True)
    cv2.imwrite("ind/a.png", doc); cv2.imwrite("ind/b.png", doc)
    open("ind/c.png","wb").write(b"corrupt-not-an-image")
    vw = cv2.VideoWriter("test.avi", cv2.VideoWriter_fourcc(*"MJPG"), 10, (320,240))
    for i in range(12):
        fr = np.zeros((240,320,3), np.uint8)
        cv2.rectangle(fr, (10+i*15,80), (50+i*15,140), (255,255,255), -1)
        vw.write(fr)
    vw.release()
    # extra assets for the "More practice" problems (E1-E8)
    canvas = np.full((400,600,3), 60, np.uint8)
    cv2.rectangle(canvas, (80,100), (220,260), (60,180,60), -1)   # green box (BGR)
    cv2.circle(canvas, (430,180), 70, (60,60,200), -1)            # red circle
    cv2.imwrite("color.jpg", canvas)
    bc = np.full((400,600,3), 200, np.uint8)
    x, rng = 150, np.random.default_rng(0)
    while x < 450:                        # random-width vertical bars = fake barcode
        bw = int(rng.integers(3, 10))
        cv2.rectangle(bc, (x,140), (x+bw,260), (0,0,0), -1)
        x += bw + int(rng.integers(3, 10))
    cv2.imwrite("barcode.jpg", bc)
    tbl = np.full((400,600,3), 255, np.uint8)
    for gx in range(50, 601, 110):        # vertical table lines
        cv2.line(tbl, (gx,40), (gx,360), (0,0,0), 2)
    for gy in range(40, 401, 80):         # horizontal table lines
        cv2.line(tbl, (50,gy), (490,gy), (0,0,0), 2)
    cv2.imwrite("lines.jpg", tbl)
    return doc

doc = make_sample_assets()
print("Sample assets ready: input.jpg, noisy.jpg, low.jpg, color.jpg, barcode.jpg, lines.jpg, ind/, test.avi")
show(doc, "synthetic input.jpg  (replace with your own image!)")


> Run the setup cell above first. Each problem: statement → reference solution → demo. For drilling: read the statement, close the solution, write it yourself, then compare.


# Easy drills — finger exercises (do these first)

One concept each, 2–5 minutes apiece. These are the kind of thing used as an
**opener** in a live screen to see if you're fluent, before any real problem.
If any of these takes you more than 5 minutes, drill it until it doesn't.

## Easy 1 — Load, resize to half, save

`imread → resize → imwrite`. The absolute baseline.

In [ ]:
img = cv2.imread("input.jpg")
half = cv2.resize(img, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA)
cv2.imwrite("e1_out.png", half)
print(img.shape, "→", half.shape)

## Easy 2 — Crop a region (array slicing)

Crop the box `(x1,y1,x2,y2)` from the image. Remember: NumPy is `[y, x]`, i.e. `img[y1:y2, x1:x2]`.

In [ ]:
x1, y1, x2, y2 = 50, 30, 300, 200
crop = img[y1:y2, x1:x2]
show(crop, f"Easy 2 - crop {crop.shape}")

## Easy 3 — Draw detections with labels

Given a list of detections, draw each box and its label. `rectangle` + `putText` — you'll write this in every CV job.

In [ ]:
dets = [
    {"box": (40, 40, 200, 180),  "label": "box",   "conf": 0.91},
    {"box": (220, 90, 380, 240), "label": "label", "conf": 0.73},
]
vis = img.copy()
for d in dets:
    x1, y1, x2, y2 = d["box"]
    cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.putText(vis, f'{d["label"]} {d["conf"]:.2f}', (x1, y1 - 6),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
show(vis, "Easy 3 - detections")

## Easy 4 — Box format conversion: xyxy ↔ xywh

Detectors output different formats (YOLO uses center-xywh). Write both directions without thinking.

In [ ]:
def xyxy_to_xywh(b):
    x1, y1, x2, y2 = b
    return (x1, y1, x2 - x1, y2 - y1)          # top-left + width/height

def xywh_to_xyxy(b):
    x, y, w, h = b
    return (x, y, x + w, y + h)

b = (40, 40, 200, 180)
print(b, "→", xyxy_to_xywh(b), "→", xywh_to_xyxy(xyxy_to_xywh(b)))
assert xywh_to_xyxy(xyxy_to_xywh(b)) == b

## Easy 5 — Rescale boxes after a resize

Model ran on a 640×640 resize; map boxes back to the original image. Just multiply by the ratio per axis.

In [ ]:
def scale_boxes(boxes, from_wh, to_wh):
    rx, ry = to_wh[0] / from_wh[0], to_wh[1] / from_wh[1]
    return [(x1 * rx, y1 * ry, x2 * rx, y2 * ry) for x1, y1, x2, y2 in boxes]

H, W = img.shape[:2]
print(scale_boxes([(100, 100, 300, 300)], (640, 640), (W, H)))

## Easy 6 — Clip boxes to image bounds

Boxes from a model can fall outside the frame. Clip before cropping or you'll get empty slices.

In [ ]:
def clip_box(box, w, h):
    x1, y1, x2, y2 = box
    return (max(0, int(x1)), max(0, int(y1)), min(w, int(x2)), min(h, int(y2)))

print(clip_box((-20, 50, 5000, 300), W, H))

## Easy 7 — Mean brightness (day / night check)

A one-liner with a decision on top — used in real pipelines to switch models or flag bad cameras.

In [ ]:
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
brightness = gray.mean()
print(f"brightness {brightness:.1f} →", "day" if brightness > 80 else "night/dark")

## Easy 8 — Percentage of mask pixels

Threshold, then compute what fraction of the image is foreground. `mask > 0` is a boolean array — use `.mean()`.

In [ ]:
_, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
pct = (mask > 0).mean() * 100
print(f"{pct:.1f}% foreground")
show(mask, "Easy 8 - mask")

## Easy 9 — Prep a frame for a model

uint8 BGR HWC → float32 RGB CHW in [0,1], plus a batch dim. Every inference pipeline has this exact function.

In [ ]:
def to_tensor(frame, size=(640, 640)):
    x = cv2.resize(frame, size)
    x = cv2.cvtColor(x, cv2.COLOR_BGR2RGB)
    x = x.astype(np.float32) / 255.0
    x = x.transpose(2, 0, 1)          # HWC → CHW
    return x[None]                    # add batch dim → (1,3,H,W)

print(to_tensor(img).shape, to_tensor(img).dtype)

## Easy 10 — Rolling FPS counter

Keep the last N frame timestamps in a `deque`; FPS = (N−1) / (newest − oldest). No cumulative averages — they hide stalls.

In [ ]:
import time

class FPS:
    def __init__(self, n=30):
        self.times = deque(maxlen=n)
    def tick(self):
        self.times.append(time.perf_counter())
        if len(self.times) < 2:
            return 0.0
        return (len(self.times) - 1) / (self.times[-1] - self.times[0])

fps = FPS()
for _ in range(10):
    time.sleep(0.01)
    v = fps.tick()
print(f"~{v:.0f} FPS")

## Easy 11 — Filter detections by confidence and class

Plain list-comprehension logic over model output. Asked constantly because it's what you do with detections all day.

In [ ]:
dets = [
    {"cls": "person", "conf": 0.92}, {"cls": "person", "conf": 0.41},
    {"cls": "truck",  "conf": 0.88}, {"cls": "person", "conf": 0.77},
]
keep = [d for d in dets if d["cls"] == "person" and d["conf"] >= 0.5]
print(len(keep), "kept:", keep)

## Easy 12 — Stack frames into a batch

Collect N frames and stack into `(N, H, W, 3)` for batched inference. `np.stack`, not `np.array` on a ragged list.

In [ ]:
frames = [cv2.resize(img, (320, 320)) for _ in range(4)]
batch = np.stack(frames)
print(batch.shape, batch.dtype)

---

# Core drills — the real problems start here

## Drill 1 [DRILL] (was P6) — Resize keeping aspect ratio, pad to square (letterbox)

**Tests:** the standard model-input preprocessing; aspect-ratio math, padding.

**The problem:** detection models want a fixed square input (e.g. 640×640) but
photos come in any shape. Naive `resize` to a square squashes objects.

**The plan:**

```text
   stretch (bad)             letterbox (good)
  +-----------+             +-----------+
  | oO -> oOO |             |###########|  <- gray pad (114)
  | distorted!|             |   image   |
  +-----------+             | untouched |
                            |###########|
                     scale by LONG side, pad the rest
```

**Why this way:** three options exist. Stretching distorts geometry — a square
carton becomes a rectangle and the model's learned shapes break. Center-cropping
keeps shapes but throws away edge pixels (where detections might be).
Letterboxing keeps *every pixel and every shape*, paying only some wasted pad.
Keep the scale and offsets — you need them to map detections back to the
original image coordinates.

*Why: Detector preprocessing — the single most likely coding ask.*


In [ ]:
import cv2
import numpy as np

def letterbox(path, size=640, out="p6_out.png"):
    img = cv2.imread(path)
    h, w = img.shape[:2]
    scale = size / max(h, w)                         # fit the LONG side
    new_w, new_h = int(w * scale), int(h * scale)
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    canvas = np.full((size, size, 3), 114, dtype=np.uint8)  # 114 = YOLO pad-gray
    top, left = (size - new_h) // 2, (size - new_w) // 2
    canvas[top:top + new_h, left:left + new_w] = resized
    cv2.imwrite(out, canvas)
    return canvas

In [ ]:
show(letterbox("input.jpg", 640), "Problem 6 - letterboxed 640")

**Watch out:** `INTER_AREA` for shrinking, `INTER_LINEAR/CUBIC` for enlarging.
Keep the scale + padding offsets if you later need to map detections back.


## Drill 2 [DRILL] (was P8) — IoU of two boxes

**Tests:** the fundamental box metric; coordinate math; the empty-overlap edge case.

**The problem:** score how much two boxes agree, as a number in [0, 1] — the
currency of all detection logic (NMS, evaluation, tracker matching).

**The plan:**

```text
   +-------+
   |   A   |            IoU = overlap / (areaA + areaB - overlap)
   |   +---+----+
   |   |###|    |       ### intersection rectangle:
   +---+---+    |       its left  = max(A.left,  B.left)
       |    B   |       its right = min(A.right, B.right)
       +--------+       (same idea for top/bottom)
```

**Why this way:** the intersection of two ranges on ONE axis is simply
[max(starts), min(ends)] — do that for x and for y, multiply the two lengths.
The `max(0, ...)` clamp is the whole edge case: for disjoint boxes min-max goes
negative and would fabricate overlap. And union must subtract the intersection,
or the shared area is counted twice.

*Why: IoU — the most-asked CV interview question anywhere.*


In [ ]:
def iou(a, b):
    """Boxes as (x1, y1, x2, y2). Returns intersection-over-union in [0,1]."""
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])     # intersection top-left
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])     # intersection bottom-right
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)   # clamp at 0 → no overlap
    inter = iw * ih
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    union = area_a + area_b - inter                 # don't double-count the overlap
    return inter / union if union > 0 else 0.0

In [ ]:
print("IoU =", iou((0,0,10,10), (5,5,15,15)))

**Watch out:** the `max(0, ...)` clamp — without it, non-overlapping boxes give a
bogus positive area.


## Drill 3 [DRILL] (was P9) — Non-Maximum Suppression (from scratch)

**Tests:** the dedup algorithm every detector needs. Sort by score, greedily keep
the best, drop high-IoU neighbours.

**The problem:** a detector fires several overlapping boxes on the *same*
object. Keep exactly one box per object.

**The plan:**

```text
 scores:  A=0.9   B=0.8   C=0.7
 1. take best remaining:  A  -> KEEP
 2. IoU(A,B)=0.8 (same object)  -> drop B
    IoU(A,C)=0.0 (elsewhere)    -> C survives
 3. take best remaining:  C  -> KEEP        result: [A, C]
```

**Why this way:** greedy-by-confidence works because the highest-score box is
usually the best-localized one — lock it in, delete everything that mostly
overlaps it. The plain-Python version shows the algorithm; the NumPy version
exists because the inner IoU loop is O(n²) — vectorizing "winner vs all
survivors" into one array pass is what makes thousands of boxes per frame
feasible. Name the follow-up: Soft-NMS decays scores instead of deleting, for
crowded scenes where two real objects genuinely overlap.

*Why: NMS from scratch — the classic follow-up to IoU.*


In [ ]:
def nms(boxes, scores, iou_thresh=0.5):
    """boxes: list of (x1,y1,x2,y2); scores: list of floats. Returns kept indices."""
    candidates = sorted(range(len(boxes)), key=lambda i: scores[i], reverse=True)  # best first
    keep = []
    while candidates:
        best = candidates.pop(0)        # highest score still in play
        keep.append(best)
        candidates = [i for i in candidates             # drop everything that
                      if iou(boxes[best], boxes[i]) < iou_thresh]  # overlaps the winner
    return keep

In [ ]:
print("keep:", nms([(0,0,10,10),(1,1,11,11),(50,50,60,60)], [0.9,0.8,0.7]))

NumPy/vectorized version (worth knowing for "make it fast"):


In [ ]:
import numpy as np

def nms_np(boxes, scores, iou_thresh=0.5):
    boxes = np.asarray(boxes, dtype=float)
    x1, y1, x2, y2 = boxes.T                # columns → four 1-D arrays
    areas = (x2 - x1) * (y2 - y1)
    order = scores.argsort()[::-1]          # indices, highest score first
    keep = []
    while order.size > 0:
        best = order[0]; keep.append(best)
        others = order[1:]
        # IoU of `best` vs ALL remaining boxes at once (no inner loop)
        inter_x1 = np.maximum(x1[best], x1[others])
        inter_y1 = np.maximum(y1[best], y1[others])
        inter_x2 = np.minimum(x2[best], x2[others])
        inter_y2 = np.minimum(y2[best], y2[others])
        inter_w = np.maximum(0, inter_x2 - inter_x1)
        inter_h = np.maximum(0, inter_y2 - inter_y1)
        intersection = inter_w * inter_h
        overlaps = intersection / (areas[best] + areas[others] - intersection)
        order = others[overlaps < iou_thresh]   # keep only low-overlap survivors
    return keep

In [ ]:
print("keep:", nms_np([(0,0,10,10),(1,1,11,11),(50,50,60,60)], np.array([0.9,0.8,0.7])))

**Watch out:** reuse `iou` from Problem 8 — interviewers love when you compose.
Soft-NMS (decay scores instead of dropping) is the follow-up for crowded scenes.


## Drill 4 [DRILL] (was P10) — Count detections inside a zone (point-in-polygon)

**Tests:** geometry on detections; using the **bottom-center** (foot) point, not
the box center.

**The problem:** given detection boxes and a zone polygon, count who is IN the
zone.

**The plan:**

```text
      zone (on the floor)          a tall object, oblique camera:
   +---------------+                 +----+
   |               |                 |    | <- box CENTER lands here
   |      x <------|---- foot        |    |    (inside the zone?!)
   +---------------+                 |    |
                                     | x  | <- foot = ((x1+x2)/2, y2)
                                     +----+    where it actually STANDS
```

**Why this way:** zones are painted on the *floor*, and objects touch the world
at their *feet* — with an angled camera, a box's geometric center can sit inside
a zone the object isn't standing in. Using the bottom-center point is the
standard fix (and a favorite interview gotcha). `cv2.pointPolygonTest` handles
arbitrary polygons; pass `False` to get just the inside/on/outside sign — we
don't need the distance, and skipping it is faster.

*Why: Zone counting — core video-analytics business logic.*


In [ ]:
import cv2
import numpy as np

def count_in_zone(boxes, polygon):
    """polygon: list of (x,y). Counts boxes whose foot point is inside."""
    polygon_pts = np.array(polygon, dtype=np.int32)
    count = 0
    for (x1, y1, x2, y2) in boxes:
        foot = (int((x1 + x2) / 2), int(y2))            # bottom-center point
        if cv2.pointPolygonTest(polygon_pts, foot, False) >= 0:  # >=0 = inside/on
            count += 1
    return count

In [ ]:
print("in zone:", count_in_zone([(100,100,200,300),(400,400,450,450)], [(50,50),(350,50),(350,350),(50,350)]))

**Watch out:** use the **foot point** (where the object meets the floor), not the
geometric center — the standard gotcha for tall objects / oblique cameras.


## Drill 5 [DRILL] (was P11) — Line-crossing counter (direction-aware)

**Tests:** tracking-style logic; cross-product sign to detect a crossing and its
direction between consecutive frames.

**The problem:** count objects crossing a virtual line — separately for each
direction (entering vs leaving).

**The plan:**

```text
             p_prev   (side +)
  A ------------ line ------------ B
             p_cur    (side -)

 side(p) = z of cross product (B-A) x (p-A)
 sign flips between consecutive frames  ==>  the track crossed
 which sign it ENDS on                  ==>  which direction
```

**Why this way:** comparing raw x or y coordinates only works for perfectly
vertical/horizontal lines; the cross-product sign works for a line at *any*
angle and gives direction for free. Counting per **track id** (not per frame)
is essential — one person crossing would otherwise be counted at every frame
near the line. In production you'd also debounce (require K of N frames on the
new side) to survive detector flicker.

*Why: Line-crossing counter — same, direction-aware.*


In [ ]:
def side(line_a, line_b, p):
    """Sign of which side of line (a->b) point p is on (cross product z)."""
    return ((line_b[0] - line_a[0]) * (p[1] - line_a[1]) -
            (line_b[1] - line_a[1]) * (p[0] - line_a[0]))   # >0 / <0 / 0=on line

def count_crossings(tracks, line_a, line_b):
    """tracks: {id: [(x,y) per frame]}. Returns (in_count, out_count)."""
    entering = leaving = 0
    for path in tracks.values():
        for prev_point, cur_point in zip(path, path[1:]):   # consecutive frames
            side_prev = side(line_a, line_b, prev_point)
            side_cur = side(line_a, line_b, cur_point)
            if side_prev == 0 or side_cur == 0:   # exactly ON the line: skip
                continue
            if (side_prev > 0) != (side_cur > 0):   # sign flipped → crossed
                if side_cur > 0: entering += 1
                else: leaving += 1
    return entering, leaving

In [ ]:
print("(in, out) =", count_crossings({1:[(0,0),(0,10)], 2:[(0,10),(0,0)]}, (-5,5), (5,5)))

**Watch out:** count **per track id** so one object is counted once per crossing,
not once per frame. Debounce flicker in real systems (K-of-N frames).


## Drill 6 [DRILL] (was P13) — Robust capture: read, sample every Nth frame, save

**Tests:** the capture loop, failure handling, frame sampling, releasing resources.

**The problem:** open a video (file or RTSP stream), keep every Nth frame as an
image, and don't crash or leak when the source misbehaves.

**The plan:**

```text
 frames:   0   1  ...  14  [15]  16 ...  29  [30]     every=15
          save                save            save

 the loop that matters:
    while True:  ok, frame = cap.read()
                 if not ok: break         <- EOF *or* dropped stream
    finally:     cap.release()            <- runs no matter what
```

**Why this way:** count-and-modulo beats seeking — jumping around with
`CAP_PROP_POS_FRAMES` is unreliable on many codecs (seeks snap to keyframes),
while sequential reading is always exact. The `ok` guard and `finally: release`
ARE the fault-tolerance answer: streams end without warning, and leaked capture
handles are how long-running services die slowly. For live RTSP, wrap the whole
thing in reconnect-with-backoff.

*Why: Robust video capture — directly from the JD.*


In [ ]:
import cv2
import os

def sample_frames(source, out_dir="frames", every=15):
    """source: a video path or RTSP url. Saves every Nth frame as an image."""
    os.makedirs(out_dir, exist_ok=True)
    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        raise RuntimeError(f"cannot open {source}")
    frame_index = saved_count = 0
    try:
        while True:
            ok, frame = cap.read()
            if not ok:                       # end of file OR dropped stream
                break
            if frame_index % every == 0:
                filename = os.path.join(out_dir, f"frame_{frame_index:05d}.jpg")
                cv2.imwrite(filename, frame)
                saved_count += 1
            frame_index += 1
    finally:
        cap.release()                        # ALWAYS release (no leaks)
    print(f"read {frame_index} frames, saved {saved_count}")
    return saved_count

In [ ]:
print("saved", sample_frames("test.avi","frames",3), "frames"); show(cv2.imread("frames/frame_00000.jpg"), "a sampled frame")

**Watch out:** check `isOpened()`, handle `ok == False` (don't assume frames keep
coming), and `release()` in a `finally`. For live RTSP you'd wrap this in a
reconnect-with-backoff loop (see the fault-tolerance chapter).


## Drill 7 [DRILL] (was P16) — Vectorize it (no pixel loops)

**Tests:** that you reach for NumPy, not `for y: for x:`. Example: threshold +
tint without loops.

**The problem:** do a per-pixel operation (select + recolor) without writing a
per-pixel Python loop.

**The plan:**

```text
 loops:  for y: for x: if gray[y,x] > 180: img[y,x] = red
         ==> 350,000 interpreted Python iterations        (slow)

 numpy:  mask = gray > 180        one C-speed comparison  -> bool array
         img[mask] = (0, 0, 255)  one C-speed scatter     (fast)
```

**Why this way:** NumPy's entire value is that the loop happens inside compiled
C, not the Python interpreter — typically ~100-1000x faster. Boolean-mask
indexing and `axis=` reductions replace 90% of pixel loops you'd be tempted to
write. The interview tell: the moment you type `for y in range(h)`, stop and
ask "what's the array expression for this?"

*Why: Vectorization — cheap NumPy fluency test.*


In [ ]:
import numpy as np
import cv2

def red_tint_bright_areas(path, out="p16_out.png", bright_threshold=180):
    img = cv2.imread(path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    bright_mask = gray > bright_threshold    # boolean array, fully vectorized
    img[bright_mask] = [0, 0, 255]           # set all bright pixels red (BGR)
    cv2.imwrite(out, img)
    return img

# Per-channel mean without loops:
def channel_means(img):
    return img.reshape(-1, 3).mean(axis=0)  # (B_mean, G_mean, R_mean)

In [ ]:
show(red_tint_bright_areas("input.jpg"), "Problem 16 - bright areas tinted")

In [ ]:
print("BGR means:", channel_means(cv2.imread("input.jpg")))

**Watch out:** boolean-mask indexing (`img[mask] = ...`) and `axis=` reductions are
the whole point. If you wrote a double `for` loop over pixels, that's the wrong
answer — say "vectorize with NumPy."


## Drill 8 [PLAUSIBLE] (was P1) — Preprocess for OCR (gray → blur → Otsu → save)

**Tests:** the fundamental chain, color conversion, thresholding, saving output.

**The problem:** an OCR engine can't read a color photo — it wants crisp
black-and-white where ink is cleanly separated from paper. Turn a messy photo
into exactly that.

**The plan:**

```text
photo (3 channels)
   |  cvtColor        drop color, keep brightness (1 channel)
   v
gray
   |  GaussianBlur    smooth away sensor noise BEFORE deciding
   v
smooth gray
   |  Otsu threshold  every pixel becomes pure 0 or 255
   v
black/white  --imwrite-->  saved
```

**Why this way:** a fixed cutoff like `127` breaks the moment lighting changes.
Otsu instead looks at the image's brightness histogram and picks the split point
that best separates "dark pixels" from "bright pixels" — zero tuning. And blur
comes *first* because thresholding is a hard yes/no decision: one speck of noise
near the cutoff flips into a black dot. Smooth before deciding, never after.

*Why: OCR preprocessing chain — KoiReader is a document-AI company.*


In [ ]:
import cv2

def preprocess(path, out="p1_out.png"):
    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)     # denoise before threshold
    # Otsu auto-picks the threshold; THRESH_BINARY_INV makes text white on black
    _, binary = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    cv2.imwrite(out, binary)
    return binary

In [ ]:
show(preprocess("input.jpg"), "Problem 1 - Otsu threshold")

**Watch out:** Otsu needs a *single-channel* image and `thresh=0`. Blur first or
threshold gets speckly.


## Drill 9 [PLAUSIBLE] (was P4) — 4-point perspective transform ("scan this label") ⭐

**Tests:** the single most likely task. Detect the document's 4 corners and warp
it to a flat top-down view. Memorize the `order_points` + `four_point_transform`
pair cold.

**The problem:** a label photographed at an angle appears as a *trapezoid*. To
read or measure it you need the flat, top-down rectangle — like a scanner sees.

**The plan:**

```text
   detected corners                          warped output
   TL._________.TR                          TL __________ TR
     /         \          3x3 perspective    |           |
    /  LABEL    \         matrix (getPersp   |  LABEL    |
   /             \        + warpPerspective) |           |
  BL._____________.BR         ==>            BL __________ BR
  find quad: Canny -> contours -> approxPolyDP == 4 points
```

**Why this way:** an affine warp (2×3 matrix: rotate/scale/shear) can *never*
fix this — affine preserves parallelism, and perspective distortion is exactly
parallel lines converging. You need the 3×3 homography, which is fully
determined by 4 point pairs — hence "4-point transform". `order_points` exists
because `getPerspectiveTransform` pairs source and destination corners *by array
position*: hand it corners in a random order and the output comes out mirrored
or rotated. The sum/diff trick identifies each corner without if-else chains.

*Why: 4-point perspective transform — flagship doc-AI geometry question.*


In [ ]:
import cv2
import numpy as np

def order_points(points):
    """Return corners as [top-left, top-right, bottom-right, bottom-left]."""
    ordered = np.zeros((4, 2), dtype="float32")
    coord_sum = points.sum(axis=1)
    ordered[0] = points[np.argmin(coord_sum)]    # top-left has smallest x+y
    ordered[2] = points[np.argmax(coord_sum)]    # bottom-right has largest x+y
    coord_diff = np.diff(points, axis=1)
    ordered[1] = points[np.argmin(coord_diff)]   # top-right has smallest y-x
    ordered[3] = points[np.argmax(coord_diff)]   # bottom-left has largest y-x
    return ordered

def four_point_transform(image, points):
    ordered = order_points(points)
    (top_left, top_right, bottom_right, bottom_left) = ordered
    width_bottom = np.linalg.norm(bottom_right - bottom_left)
    width_top = np.linalg.norm(top_right - top_left)
    out_width = int(max(width_bottom, width_top))    # output = longest measured side
    height_right = np.linalg.norm(top_right - bottom_right)
    height_left = np.linalg.norm(top_left - bottom_left)
    out_height = int(max(height_right, height_left))
    # destination corners of a flat, axis-aligned rectangle (same order!)
    destination = np.array([[0, 0], [out_width - 1, 0],
                            [out_width - 1, out_height - 1], [0, out_height - 1]],
                           dtype="float32")
    perspective_matrix = cv2.getPerspectiveTransform(ordered, destination)
    return cv2.warpPerspective(image, perspective_matrix, (out_width, out_height))

def scan(path, out="p4_out.png"):
    img = cv2.imread(path)
    ratio = img.shape[0] / 500.0
    resized = cv2.resize(img, (int(img.shape[1] / ratio), 500))
    gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(cv2.GaussianBlur(gray, (5, 5), 0), 75, 200)
    contours, _ = cv2.findContours(edges, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)[:5]
    document_corners = None
    for contour in contours:
        perimeter = cv2.arcLength(contour, True)
        corners = cv2.approxPolyDP(contour, 0.02 * perimeter, True)  # simplify
        if len(corners) == 4:                              # found a quadrilateral
            document_corners = corners.reshape(4, 2) * ratio  # scale to full res
            break
    if document_corners is None:
        raise RuntimeError("no 4-corner document found")
    warped = four_point_transform(img, document_corners.astype("float32"))
    cv2.imwrite(out, warped)
    return warped

In [ ]:
show(scan("input.jpg"), "Problem 4 - perspective-corrected")

**Watch out:** `getPerspectiveTransform` needs **float32** points in a consistent
order — that's why `order_points` exists. Scale corners back up if you detected on
a resized copy.


## Drill 10 [PLAUSIBLE] (was P12) — Sort OCR boxes into reading order

**Tests:** turning detections into text order — group into rows, then left-to-right.

**The problem:** OCR returns word boxes in arbitrary order; reconstruct the
order a human would read them in.

**The plan:**

```text
 naive sort by (y, x) fails:  same-line words differ by a few px in y,
                              so the sort interleaves lines:
                              word(y=10) word(y=12) word(y=11) ...

 fix: 1. sort by y      2. group into ROWS with a tolerance
      3. sort each row by x     4. concatenate rows
```

**Why this way:** the tolerance (about half the text height) is what makes row
grouping robust to slightly wavy or tilted scans; the running-average row
anchor keeps a long row from drifting. A single lexicographic sort has no such
slack — it's the bug interviewers expect you to know about. This exact "logic
on top of an OCR model" is KoiReader's bread and butter.

*Why: OCR reading order — very on-brand for KoiReader.*


In [ ]:
def reading_order(boxes, row_tol=15):
    """boxes: list of (x1,y1,x2,y2). Returns indices in reading order."""
    order = list(range(len(boxes)))
    order.sort(key=lambda i: boxes[i][1])         # by top y first
    rows, current_row, current_row_y = [], [], None
    for i in order:
        box_center_y = (boxes[i][1] + boxes[i][3]) / 2
        if current_row_y is None or abs(box_center_y - current_row_y) <= row_tol:
            current_row.append(i)                 # same line of text
            current_row_y = (box_center_y if current_row_y is None
                             else (current_row_y + box_center_y) / 2)  # running avg
        else:
            rows.append(current_row)              # row finished, start a new one
            current_row, current_row_y = [i], box_center_y
    if current_row:
        rows.append(current_row)
    result = []
    for row in rows:
        row.sort(key=lambda i: boxes[i][0])       # within a row, left to right
        result.extend(row)
    return result

In [ ]:
print("order:", reading_order([(100,10,140,30),(10,12,50,32),(10,80,60,100)]))

**Watch out:** `row_tol` groups boxes into the same line; tune to text height.
This is exactly the kind of "logic on top of an OCR model" KoiReader builds.


## Drill 11 [PLAUSIBLE] (was P14) — Motion detection via frame differencing, write annotated video

**Tests:** per-frame processing, background diff, contours on a mask, `VideoWriter`.

**The problem:** find and box the *moving* things in a video, writing an
annotated copy.

**The plan:**

```text
 prev frame    cur frame     absdiff      threshold     dilate+contours
 [.......]  vs [...o...] ==> [...#...] ==> [...#...] ==> [..[box]..]
 (blur both first - otherwise sensor noise "moves" in every pixel)
```

**Why this way:** frame differencing needs zero training and one frame of
state — the right first answer. Know its weakness out loud: it detects *change*,
so an object that stops moving disappears. The production upgrade is
`cv2.createBackgroundSubtractorMOG2()`, which learns the background over time
and tolerates gradual lighting change. Practical trap: `VideoWriter` silently
produces an empty file if the frame size doesn't match — always pass the real
width/height.

*Why: Frame differencing + annotated video write.*


In [ ]:
import cv2

def detect_motion(source, out="p14_out.mp4", min_area=800):
    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        raise RuntimeError(f"cannot open {source}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 25            # some sources report 0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")         # codec for .mp4
    writer = cv2.VideoWriter(out, fourcc, fps, (width, height))
    prev_gray = None                                 # nothing to diff on frame 1
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            gray = cv2.GaussianBlur(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY),
                                    (21, 21), 0)
            if prev_gray is not None:
                frame_diff = cv2.absdiff(prev_gray, gray)   # |previous - current|
                motion_mask = cv2.threshold(frame_diff, 25, 255,
                                            cv2.THRESH_BINARY)[1]  # keep big changes
                motion_mask = cv2.dilate(motion_mask, None, iterations=2)  # merge blobs
                contours, _ = cv2.findContours(motion_mask, cv2.RETR_EXTERNAL,
                                               cv2.CHAIN_APPROX_SIMPLE)
                for contour in contours:
                    if cv2.contourArea(contour) < min_area:  # ignore tiny flicker
                        continue
                    x, y, box_w, box_h = cv2.boundingRect(contour)
                    cv2.rectangle(frame, (x, y), (x + box_w, y + box_h),
                                  (0, 255, 0), 2)
            writer.write(frame)
            prev_gray = gray                 # current frame becomes the baseline
    finally:
        cap.release(); writer.release()      # ALWAYS release both (no leaks)
    return out

In [ ]:
detect_motion("test.avi","motion.avi"); print("wrote motion.avi")

**Watch out:** `VideoWriter` size must match the frames you write, or you get an
empty file. Blur before diff to kill sensor noise. `fourcc` "mp4v" for `.mp4`.


## Drill 12 [PLAUSIBLE] (was P17) — Fault-tolerant batch image processor

**Tests:** the "manage a long-running job, don't crash on one bad input" skill.
Process a folder, skip and log corrupt files, keep going.

**The problem:** process a folder of 10,000 images where file #37 is corrupt —
without the whole job dying at #37.

**The plan:**

```text
 for each file:
     try:     read -> check None -> process -> write   ok += 1
     except:  log the name and reason, fail += 1, CONTINUE

 one bad file = one warning line, not a dead 3-hour batch job
```

**Why this way:** the try/except goes *inside* the loop, per file — wrap the
whole loop instead and the first bad file still kills everything after it. The
subtle trap this problem is really testing: `cv2.imread` does NOT raise on
failure, it silently returns `None` — you must check and raise yourself.
Returning (ok, fail) counts and logging skips makes the job observable, which
is what "fault-tolerant" means in practice.

*Why: Fault-tolerant batch processor — matches the crash-proof theme.*


In [ ]:
import os
import cv2
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

def process_folder(in_dir, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    exts = (".jpg", ".jpeg", ".png", ".bmp")
    ok = fail = 0
    for name in sorted(os.listdir(in_dir)):
        if not name.lower().endswith(exts):     # skip non-image files
            continue
        src = os.path.join(in_dir, name)
        try:
            img = cv2.imread(src)
            if img is None:
                raise ValueError("unreadable / corrupt")
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            cv2.imwrite(os.path.join(out_dir, name), gray)
            ok += 1
        except Exception as e:                      # one bad file can't kill the run
            logging.warning("skipped %s: %s", name, e)
            fail += 1
    logging.info("done: %d ok, %d failed", ok, fail)
    return ok, fail

In [ ]:
print("(ok, failed) =", process_folder("ind","outd"))

**Watch out:** `cv2.imread` returns **None** on failure (it doesn't raise) — guard
it. Catch per-file so the batch continues; log what you skipped. That *is* fault
tolerance.
